# Disproportionality Analysis — FEARS Pipeline Output
## PRR, ROR, and EBGM Signal Detection

Based on methodology from:
- **Evans et al. (2001)** — PRR
- **van Puijenbroek et al. (2002)** — ROR
- **DuMouchel (1999)** — MGPS / EBGM

**Data:** Final pipeline output (adult + pediatric cohorts)
**Subgroups:** Adult, Pediatric (overall), NICHD age bands

## 0. Setup

In [198]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from IPython.display import display, Markdown
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (12, 5), "figure.dpi": 120, "font.size": 11})

## 1. Load Data

In [199]:
DATA_ROOT = Path("../data/output")

adult = pl.read_parquet(DATA_ROOT / "Adult" / "patient_report_reporter_drug_reaction_full_data.parquet")
pediatric = pl.read_parquet(DATA_ROOT / "Pediatric" / "patient_report_reporter_drug_reaction_full_data.parquet")

print(f"Adult:      {adult.height:>12,} rows | {adult['safetyreportid'].n_unique():>10,} unique reports")
print(f"Pediatric:  {pediatric.height:>12,} rows | {pediatric['safetyreportid'].n_unique():>10,} unique reports")
print(f"Total:      {adult.height + pediatric.height:>12,} rows")

Adult:        17,813,299 rows |  2,506,379 unique reports
Pediatric:       953,037 rows |    221,182 unique reports
Total:        18,766,336 rows


## 2. Core Functions

### 2.1 Precompute Pair Counts

สร้าง lookup table ครั้งเดียว แล้วใช้ O(1) ต่อ pair — เร็วกว่า filter ซ้ำหลายรอบ

In [200]:
def precompute_pair_counts(df: pl.DataFrame, drug_col: str, event_col: str) -> tuple:
    """Precompute all drug-event pair counts in one pass.

    Returns:
        pair_lookup: dict {(drug, event): count}
        drug_totals: dict {drug: total_events}
        event_totals: dict {event: total_drugs}
        N: total drug-event pairs
    """
    pairs = df.select(
        pl.col(drug_col).alias("drug"),
        pl.col(event_col).alias("event"),
    )

    N = pairs.height

    pair_counts = pairs.group_by(["drug", "event"]).agg(pl.len().alias("count"))
    drug_totals = dict(pairs.group_by("drug").agg(pl.len().alias("total")).iter_rows())
    event_totals = dict(pairs.group_by("event").agg(pl.len().alias("total")).iter_rows())

    pair_lookup = {}
    for row in pair_counts.iter_rows(named=True):
        pair_lookup[(row["drug"], row["event"])] = row["count"]

    return pair_lookup, drug_totals, event_totals, N


def get_2x2(pair_lookup, drug_totals, event_totals, N, target_drug, target_event):
    """Get 2x2 table values from precomputed counts (O(1) lookup).

    Returns: a, b, c, d
    """
    a = pair_lookup.get((target_drug, target_event), 0)
    b = drug_totals.get(target_drug, 0) - a      # drug + other events
    c = event_totals.get(target_event, 0) - a     # other drugs + this event
    d = N - a - b - c                             # other drugs + other events
    return a, b, c, d

print("Defined: precompute_pair_counts(), get_2x2()")

Defined: precompute_pair_counts(), get_2x2()


### 2.2 PRR (Proportional Reporting Ratio)

$$PRR = \frac{a / (a+b)}{c / (c+d)}$$

$$SE(\ln PRR) = \sqrt{\frac{1}{a} - \frac{1}{a+b} + \frac{1}{c} - \frac{1}{c+d}}$$

Signal: 95% CI lower limit > 1

In [201]:
def compute_prr(a, b, c, d):
    """PRR with 95% CI and chi-squared.

    Formula (from image):
        PRR = (a/(a+b)) / (c/(c+d))
        SE(lnPRR) = sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))
        95% CI = exp(ln(PRR) +/- 1.96 * SE)
    Signal: 95% CI lower limit > 1
    """
    if a == 0 or c == 0 or (a + b) == 0 or (c + d) == 0:
        return {"PRR": np.nan, "PRR_lower": np.nan, "PRR_upper": np.nan,
                "chi2": np.nan, "PRR_signal": False}

    prr = (a / (a + b)) / (c / (c + d))

    se = np.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))
    ci_lo = np.exp(np.log(prr) - 1.96 * se)
    ci_hi = np.exp(np.log(prr) + 1.96 * se)

    table = np.array([[a, b], [c, d]])
    chi2_val, _, _, _ = chi2_contingency(table, correction=False)

    signal = ci_lo > 1

    return {
        "PRR": round(prr, 2),
        "PRR_lower": round(ci_lo, 2),
        "PRR_upper": round(ci_hi, 2),
        "chi2": round(chi2_val, 2),
        "PRR_signal": signal,
    }

print("Defined: compute_prr()")

Defined: compute_prr()


### 2.3 ROR (Reporting Odds Ratio)

$$ROR = \frac{a/c}{b/d} = \frac{a \times d}{b \times c}$$

$$SE(\ln ROR) = \sqrt{\frac{1}{a} + \frac{1}{b} + \frac{1}{c} + \frac{1}{d}}$$

Signal: 95% CI lower limit > 1

In [202]:
def compute_ror(a, b, c, d):
    """ROR with 95% CI.

    Formula (from image):
        ROR = (a/c) / (b/d) = (a*d) / (b*c)
        SE(lnROR) = sqrt(1/a + 1/b + 1/c + 1/d)
        95% CI = exp(ln(ROR) +/- 1.96 * SE)
    Signal: 95% CI lower limit > 1
    """
    if a == 0 or b == 0 or c == 0 or d == 0:
        return {"ROR": np.nan, "ROR_lower": np.nan, "ROR_upper": np.nan,
                "ROR_signal": False}

    ror = (a * d) / (b * c)

    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    ci_lo = np.exp(np.log(ror) - 1.96 * se)
    ci_hi = np.exp(np.log(ror) + 1.96 * se)

    signal = ci_lo > 1

    return {
        "ROR": round(ror, 2),
        "ROR_lower": round(ci_lo, 2),
        "ROR_upper": round(ci_hi, 2),
        "ROR_signal": signal,
    }

print("Defined: compute_ror()")

Defined: compute_ror()


### 2.4 EBGM (Empirical Bayesian Geometric Mean)

Frequentist observed/expected ratio with log-normal CI:

$$EBGM = \frac{a \times (a+b+c+d)}{(a+c) \times (a+b)}$$

$$SE(\ln EBGM) = \sqrt{\frac{1}{a} + \frac{1}{b} + \frac{1}{c} + \frac{1}{d}}$$

Signal: EBGM05 (lower 95% CI) > 2

In [203]:
def compute_ebgm(a, b, c, d):
    """EBGM with 95% CI (frequentist, from image).

    Formula (from image):
        EBGM = a(a+b+c+d) / ((a+c)(a+b))
        SE(lnEBGM) = sqrt(1/a + 1/b + 1/c + 1/d)
        95% CI = exp(ln(EBGM) +/- 1.96 * SE)
    Signal: EBGM05 (lower 95% CI) > 2
    """
    N = a + b + c + d

    if a == 0 or b == 0 or c == 0 or d == 0 or N == 0:
        return {"EBGM": np.nan, "EBGM05": np.nan, "EBGM95": np.nan,
                "EBGM_signal": False}

    ebgm = (a * N) / ((a + c) * (a + b))

    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    ebgm05 = np.exp(np.log(ebgm) - 1.96 * se)  # lower 95% CI
    ebgm95 = np.exp(np.log(ebgm) + 1.96 * se)  # upper 95% CI

    signal = ebgm05 > 2

    return {
        "EBGM": round(ebgm, 2),
        "EBGM05": round(ebgm05, 2),
        "EBGM95": round(ebgm95, 2),
        "EBGM_signal": signal,
    }

print("Defined: compute_ebgm()")

Defined: compute_ebgm()


### 2.5 Batch Computation

รวม PRR + ROR + EBGM เข้าด้วยกัน คำนวณทีเดียวหลาย pairs

In [204]:
def compute_signals_batch(df, drug_col, event_col, pairs, label=""):
    """Compute PRR, ROR, EBGM for a list of (drug, event) pairs.

    Precomputes pair counts once, then O(1) per pair.
    """
    pair_lookup, drug_totals, event_totals, N = precompute_pair_counts(df, drug_col, event_col)

    results = []
    for drug, event in pairs:
        a, b, c, d = get_2x2(pair_lookup, drug_totals, event_totals, N, drug, event)
        E = ((a + b) * (a + c)) / N if N > 0 else 0

        prr = compute_prr(a, b, c, d)
        ror = compute_ror(a, b, c, d)
        ebgm = compute_ebgm(a, b, c, d)

        results.append({
            "drug": drug, "event": event,
            "N_total": N, "a": a, "b": b, "c": c, "d": d,
            "E (expected)": round(E, 2),
            **prr, **ror, **ebgm,
        })

    return pl.DataFrame(results).cast({
        "PRR_signal": pl.Boolean,
        "ROR_signal": pl.Boolean,
        "EBGM_signal": pl.Boolean,
    }, strict=False)


def _fmt(val, lo, hi):
    """Format value with 95% CI: '12.34 (10.00, 14.50)'"""
    if val is None or np.isnan(val):
        return ""
    if lo is None or np.isnan(lo):
        return f"{val:.2f}"
    return f"{val:.2f} ({lo:.2f}, {hi:.2f})"


def _fmt_ebgm(val, lo, hi):
    """Format EBGM with 95% CI: '12.34 (10.00, 14.50)'"""
    if val is None or np.isnan(val):
        return ""
    if lo is None or np.isnan(lo):
        return f"{val:.2f}"
    return f"{val:.2f} ({lo:.2f}, {hi:.2f})"


def _signal_icon(prr_sig, ror_sig, ebgm_sig):
    """Show signal status per method: 1 = signal, 0 = non-signal."""
    p = 1 if prr_sig else 0
    r = 1 if ror_sig else 0
    e = 1 if ebgm_sig else 0
    return f"{p}/{r}/{e}"


def format_signal_table(signals_df):
    """Format signals into compact publication-style table.

    Signal column shows PRR/ROR/EBGM as +/- (e.g. +/+/+ = all agree).
    """
    rows = []
    for r in signals_df.iter_rows(named=True):
        rows.append({
            "Drug": r["drug"],
            "Event": r["event"],
            "a": r["a"],
            "b": r["b"],
            "c": r["c"],
            "d": r["d"],
            "Expected (E)": round(r["E (expected)"], 2),
            "ROR (95% CI)": _fmt(r["ROR"], r["ROR_lower"], r["ROR_upper"]),
            "PRR (95% CI)": _fmt(r["PRR"], r["PRR_lower"], r["PRR_upper"]),
            "chi2": round(r["chi2"], 2) if r["chi2"] is not None and not np.isnan(r["chi2"]) else None,
            "EBGM (95% CI)": _fmt_ebgm(r["EBGM"], r["EBGM05"], r["EBGM95"]),
            "Signal (P/R/E)": _signal_icon(r["PRR_signal"], r["ROR_signal"], r["EBGM_signal"]),
        })
    return pl.DataFrame(rows).sort("a", descending=True)


def format_signal_table_by_ror(signals_df):
    """Format signals sorted by ROR descending.
    
    Filters out rows where any of a,b,c,d = 0 (ROR/EBGM undefined).
    Remaining rows sorted by numeric ROR from high to low.
    """
    sorted_df = (
        signals_df
        .filter(
            (pl.col("a") > 0) & (pl.col("b") > 0)
            & (pl.col("c") > 0) & (pl.col("d") > 0)
        )
        .sort("ROR", descending=True, nulls_last=True)
    )
    
    rows = []
    for r in sorted_df.iter_rows(named=True):
        rows.append({
            "Drug": r["drug"],
            "Event": r["event"],
            "a": r["a"],
            "b": r["b"],
            "c": r["c"],
            "d": r["d"],
            "Expected (E)": round(r["E (expected)"], 2),
            "ROR (95% CI)": _fmt(r["ROR"], r["ROR_lower"], r["ROR_upper"]),
            "PRR (95% CI)": _fmt(r["PRR"], r["PRR_lower"], r["PRR_upper"]),
            "chi2": round(r["chi2"], 2) if r["chi2"] is not None and not np.isnan(r["chi2"]) else None,
            "EBGM (95% CI)": _fmt_ebgm(r["EBGM"], r["EBGM05"], r["EBGM95"]),
            "Signal (P/R/E)": _signal_icon(r["PRR_signal"], r["ROR_signal"], r["EBGM_signal"]),
        })
    return pl.DataFrame(rows)

print("Defined: compute_signals_batch(), _fmt(), _fmt_ebgm(), _signal_icon(), format_signal_table()")

Defined: compute_signals_batch(), _fmt(), _fmt_ebgm(), _signal_icon(), format_signal_table()


## 3. Compute All Drug-Event Pairs

Compute PRR, ROR, EBGM for **every unique (drug, event) pair** in each cohort, then show Top 50 sorted by ROR.

In [205]:
def get_all_drug_event_pairs(df):
    """Get ALL unique (drug, event) pairs in the dataset."""
    pairs = (
        df.select("medicinal_product", "reaction_meddrapt")
        .unique()
        .filter(
            pl.col("medicinal_product").is_not_null()
            & pl.col("reaction_meddrapt").is_not_null()
        )
    )
    return list(zip(
        pairs["medicinal_product"].to_list(),
        pairs["reaction_meddrapt"].to_list(),
    ))

adult_pairs = get_all_drug_event_pairs(adult)
pediatric_pairs = get_all_drug_event_pairs(pediatric)

print(f"Adult pairs (all):     {len(adult_pairs):,}")
print(f"Pediatric pairs (all): {len(pediatric_pairs):,}")

Adult pairs (all):     1,187,842
Pediatric pairs (all): 224,741


## 4. Top 50 Signals (by ROR)

### 4.1 Adult Cohort

In [206]:
print("Computing Adult signals (all pairs)...")
adult_signals = compute_signals_batch(adult, "medicinal_product", "reaction_meddrapt", adult_pairs, label="Adult")
print(f"Done: {adult_signals.height:,} pairs computed")

display(Markdown("**Top 50 Adult Signals (sorted by ROR, descending):**"))
display(format_signal_table_by_ror(adult_signals).head(50))

Computing Adult signals (all pairs)...
Done: 1,187,842 pairs computed


**Top 50 Adult Signals (sorted by ROR, descending):**

Drug,Event,a,b,c,d,Expected (E),ROR (95% CI),PRR (95% CI),chi2,EBGM (95% CI),Signal (P/R/E)
str,str,i64,i64,i64,i64,f64,str,str,f64,str,str
"""VENLAFAXIN HEUMANN""","""Nosophobia""",1,4,1,17813293,0.0,"""4453323.25 (235428.17, 8423838…","""3562658.80 (256889.62, 4940852…",1781328.6,"""1781329.90 (94171.30, 33695363…","""1/1/1"""
"""ZONISAMIDE CAPS 25MG""","""Cerebrosclerosis""",1,1,5,17813292,0.0,"""3562658.40 (194632.36, 6521287…","""1781329.70 (345593.01, 9181712…",1.4844e6,"""1484441.58 (81096.85, 27172040…","""1/1/1"""
"""MYCOPHENOLATE MOFETIL\MYCOPHEN…","""Autopsy""",1,2,3,17813293,0.0,"""2968882.17 (208948.58, 4218387…","""1979255.11 (278794.76, 1405137…",1.4844e6,"""1484441.58 (104474.33, 2109194…","""1/1/1"""
"""ASPIRIN LOW DOSE""","""Hyperthecosis""",1,3,2,17813293,0.0,"""2968882.17 (208948.58, 4218387…","""2226661.88 (248865.42, 1992250…",1.4844e6,"""1484441.58 (104474.33, 2109194…","""1/1/1"""
"""FLUTICASONE PROPIONATE + SALME…","""Troponin I""",1,1,6,17813291,0.0,"""2968881.83 (165819.32, 5315580…","""1484441.42 (299603.70, 7354936…",1.2724e6,"""1272378.50 (71065.45, 22781069…","""1/1/1"""
…,…,…,…,…,…,…,…,…,…,…,…
"""SEROPHENE""","""Superovulation""",1,21,2,17813275,0.0,"""424125.60 (37029.84, 4857771.6…","""404847.20 (38078.92, 4304251.8…",269896.85,"""269898.47 (23564.48, 3091313.3…","""1/1/1"""
"""DEXYCU""","""Corectopia""",3,21,6,17813269,0.0,"""424125.45 (99444.93, 1808864.5…","""371109.90 (98452.29, 1398876.1…",742216.17,"""247406.93 (58009.64, 1055172.7…","""1/1/1"""
"""BASILIXIMAB 20 MG""","""Schistosomiasis""",2,2,43,17813252,0.0,"""414261.67 (57044.87, 3008381.8…","""207131.34 (74350.11, 577045.40…",395848.18,"""197925.54 (27254.84, 1437341.7…","""1/1/1"""


In [207]:
signal_counts = (
    adult_signals
    .filter((pl.col("a") > 0) & (pl.col("b") > 0) & (pl.col("c") > 0) & (pl.col("d") > 0))
    .with_columns(
        (pl.col("PRR_signal").cast(pl.Int8).cast(pl.Utf8) + "/" +
         pl.col("ROR_signal").cast(pl.Int8).cast(pl.Utf8) + "/" +
         pl.col("EBGM_signal").cast(pl.Int8).cast(pl.Utf8)).alias("pattern")
    )
    .group_by("pattern")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)
display(Markdown("**Signal pattern distribution (Adult):**"))
display(signal_counts)

**Signal pattern distribution (Adult):**

pattern,count
str,u32
"""0/0/0""",597868
"""1/1/1""",385738
"""1/1/0""",182235
"""1/0/0""",5360


In [208]:
display(format_signal_table_by_ror(
    adult_signals.filter(
        (pl.col("EBGM_signal") == False)
        & (pl.col("PRR_signal") == True)
        & (pl.col("ROR_signal") == True)
    )
))

Drug,Event,a,b,c,d,Expected (E),ROR (95% CI),PRR (95% CI),chi2,EBGM (95% CI),Signal (P/R/E)
str,str,i64,i64,i64,i64,f64,str,str,f64,str,str
"""TACROLIMUS CAP 5MG ACCORD HEAL…","""Pneumonia""",1,1,284805,17528492,0.03,"""61.55 (3.85, 984.01)""","""31.27 (7.82, 125.05)""",29.78,"""31.27 (1.96, 500.00)""","""1/1/0"""
"""SANDIMMUN SIM+SOLINF""","""Pneumonia""",1,1,284805,17528492,0.03,"""61.55 (3.85, 984.01)""","""31.27 (7.82, 125.05)""",29.78,"""31.27 (1.96, 500.00)""","""1/1/0"""
"""CC-90007""","""Pneumonia""",1,1,284805,17528492,0.03,"""61.55 (3.85, 984.01)""","""31.27 (7.82, 125.05)""",29.78,"""31.27 (1.96, 500.00)""","""1/1/0"""
"""MK-0653""","""Pneumonia""",1,1,284805,17528492,0.03,"""61.55 (3.85, 984.01)""","""31.27 (7.82, 125.05)""",29.78,"""31.27 (1.96, 500.00)""","""1/1/0"""
"""CIPROXAN-I.V.300""","""Pneumonia""",1,1,284805,17528492,0.03,"""61.55 (3.85, 984.01)""","""31.27 (7.82, 125.05)""",29.78,"""31.27 (1.96, 500.00)""","""1/1/0"""
…,…,…,…,…,…,…,…,…,…,…,…
"""OXYCODONE TEREPHTHALATE""","""Fatigue""",1484,51562,469146,17291107,1401.48,"""1.06 (1.01, 1.12)""","""1.06 (1.01, 1.11)""",5.01,"""1.06 (1.01, 1.12)""","""1/1/0"""
"""COSENTYX""","""Fatigue""",3390,118896,467240,17223773,3230.81,"""1.05 (1.02, 1.09)""","""1.05 (1.02, 1.09)""",8.11,"""1.05 (1.01, 1.09)""","""1/1/0"""
"""ENBREL""","""Stomatitis""",2642,294176,149533,17366948,2535.65,"""1.04 (1.00, 1.08)""","""1.04 (1.00, 1.08)""",4.58,"""1.04 (1.00, 1.08)""","""1/1/0"""


### 4.2 Pediatric Cohort — Top 50 Signals

In [209]:
print("Computing Pediatric signals (all pairs)...")
ped_signals = compute_signals_batch(pediatric, "medicinal_product", "reaction_meddrapt", pediatric_pairs, label="Pediatric")
print(f"Done: {ped_signals.height:,} pairs computed")

print("\nComputing Intersection signals (Adult cohort)...")
inter_adult_signals = compute_signals_batch(adult, "medicinal_product", "reaction_meddrapt", intersection_pairs, label="Inter-Adult")
print(f"Done: {inter_adult_signals.height:,} pairs (adult side)")

print("Computing Intersection signals (Pediatric cohort)...")
inter_ped_signals = compute_signals_batch(pediatric, "medicinal_product", "reaction_meddrapt", intersection_pairs, label="Inter-Ped")
print(f"Done: {inter_ped_signals.height:,} pairs (pediatric side)")

# Merge intersection: Adult metrics + Pediatric metrics side by side
inter_merged = (
    inter_adult_signals
    .rename({c: f"{c}_adult" for c in inter_adult_signals.columns if c not in ("drug", "event", "rxcui")})
    .join(
        inter_ped_signals
        .rename({c: f"{c}_ped" for c in inter_ped_signals.columns if c not in ("drug", "event", "rxcui")}),
        on=["drug", "event"],
        how="inner",
        suffix="_ped2",
    )
    .with_columns([
        # ROR ratio: ped / adult — >1 = stronger in pediatric
        pl.when((pl.col("ROR_adult") > 0) & (pl.col("ROR_ped") > 0))
          .then(pl.col("ROR_ped") / pl.col("ROR_adult"))
          .otherwise(None)
          .alias("ROR_ratio_ped_vs_adult"),
        # Direction
        pl.when(pl.col("ROR_ped") > pl.col("ROR_adult"))
          .then(pl.lit("Stronger in Pediatric"))
          .when(pl.col("ROR_ped") < pl.col("ROR_adult"))
          .then(pl.lit("Stronger in Adult"))
          .otherwise(pl.lit("Equal"))
          .alias("direction"),
    ])
)
print(f"\nIntersection merged: {inter_merged.height:,} pairs x {inter_merged.width} columns")

display(Markdown("**Top 50 Pediatric Signals (sorted by ROR, descending):**"))
display(format_signal_table_by_ror(ped_signals).head(50))

Computing Pediatric signals (all pairs)...
Done: 224,741 pairs computed

Computing Intersection signals (Adult cohort)...
Done: 17,157 pairs (adult side)
Computing Intersection signals (Pediatric cohort)...
Done: 17,157 pairs (pediatric side)

Intersection merged: 17,157 pairs x 42 columns


**Top 50 Pediatric Signals (sorted by ROR, descending):**

Drug,Event,a,b,c,d,Expected (E),ROR (95% CI),PRR (95% CI),chi2,EBGM (95% CI),Signal (P/R/E)
str,str,i64,i64,i64,i64,f64,str,str,f64,str,str
"""OESCLIM""","""Lymphoplasia""",1,1,2,953033,0.0,"""476516.50 (21487.46, 10567463.…","""238258.75 (33560.79, 1691475.0…",158838.33,"""158839.50 (7162.52, 3522502.72…","""1/1/1"""
"""PYRIDOSTIGMINE BROMIDE.""","""Thymoma""",1,2,1,953033,0.0,"""476516.50 (21487.46, 10567463.…","""317678.00 (25297.11, 3989362.0…",158838.33,"""158839.50 (7162.52, 3522502.72…","""1/1/1"""
"""WEZLANA""","""Plague""",1,2,1,953033,0.0,"""476516.50 (21487.46, 10567463.…","""317678.00 (25297.11, 3989362.0…",158838.33,"""158839.50 (7162.52, 3522502.72…","""1/1/1"""
"""BACLOFEN INTRATHECAL 500MCG/ML""","""Fluctuance""",1,3,1,953032,0.0,"""317677.33 (15912.20, 6342233.8…","""238258.25 (17823.53, 3184946.2…",119128.37,"""119129.62 (5967.11, 2378350.15…","""1/1/1"""
"""MILTEFOSINE""","""Leishmaniasis""",1,1,3,953032,0.0,"""317677.33 (15912.20, 6342233.8…","""158839.17 (26540.35, 950623.47…",119128.37,"""119129.62 (5967.11, 2378350.15…","""1/1/1"""
…,…,…,…,…,…,…,…,…,…,…,…
"""EVOLOCUMAB""","""Arteriosclerosis""",2,2,17,953016,0.0,"""56059.76 (7460.29, 421256.84)""","""28030.38 (9431.75, 83303.97)""",50157.05,"""25079.92 (3337.57, 188461.16)""","""1/1/1"""
"""ACENOCOUMAROL""","""Scleromalacia""",1,6,3,953027,0.0,"""52945.94 (4800.74, 583924.83)""","""45382.38 (5347.29, 385159.81)""",34035.43,"""34037.04 (3086.22, 375384.19)""","""1/1/1"""
"""AMPHOTERICIN B DEOXYCHOLATE""","""Sporotrichosis""",1,6,3,953027,0.0,"""52945.94 (4800.74, 583924.83)""","""45382.38 (5347.29, 385159.81)""",34035.43,"""34037.04 (3086.22, 375384.19)""","""1/1/1"""


In [210]:
signal_counts_ped = (
    ped_signals
    .filter((pl.col("a") > 0) & (pl.col("b") > 0) & (pl.col("c") > 0) & (pl.col("d") > 0))
    .with_columns(
        (pl.col("PRR_signal").cast(pl.Int8).cast(pl.Utf8) + "/" +
         pl.col("ROR_signal").cast(pl.Int8).cast(pl.Utf8) + "/" +
         pl.col("EBGM_signal").cast(pl.Int8).cast(pl.Utf8)).alias("pattern")
    )
    .group_by("pattern")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)
display(Markdown("**Signal pattern distribution (Adult):**"))
display(signal_counts_ped)

**Signal pattern distribution (Adult):**

pattern,count
str,u32
"""0/0/0""",101544
"""1/1/1""",80776
"""1/1/0""",36646
"""1/0/0""",1254


## 5. Intersection Dataset — Shared Drug-AE Pairs

Drug-AE pairs present in **both** Adult and Pediatric datasets.
Metrics computed separately per cohort, displayed side by side.
Sorted by `ROR_ratio_ped_vs_adult` (>1 = stronger signal in pediatric).

In [211]:
# Display Intersection — top 50 by ROR ratio
if inter_merged.height > 0:
    inter_display = []
    inter_sorted = inter_merged.sort("ROR_ratio_ped_vs_adult", descending=True, nulls_last=True)
    for r in inter_sorted.head(50).iter_rows(named=True):
        inter_display.append({
            "Drug": r["drug"],
            "Event": r["event"],
            "a (adult)": r["a_adult"],
            "a (ped)": r["a_ped"],
            "ROR adult": round(r["ROR_adult"], 2) if r["ROR_adult"] is not None and not np.isnan(r.get("ROR_adult", float("nan"))) else None,
            "ROR ped": round(r["ROR_ped"], 2) if r["ROR_ped"] is not None and not np.isnan(r.get("ROR_ped", float("nan"))) else None,
            "ROR ratio": round(r["ROR_ratio_ped_vs_adult"], 2) if r["ROR_ratio_ped_vs_adult"] is not None else None,
            "EBGM adult": round(r["EBGM_adult"], 2) if r["EBGM_adult"] is not None and not np.isnan(r.get("EBGM_adult", float("nan"))) else None,
            "EBGM ped": round(r["EBGM_ped"], 2) if r["EBGM_ped"] is not None and not np.isnan(r.get("EBGM_ped", float("nan"))) else None,
            "Direction": r["direction"],
        })
    display(Markdown(f"**Intersection: {inter_merged.height:,} shared pairs (top 50 by ROR ratio)**"))
    display(pl.DataFrame(inter_display))

    stronger_ped = inter_merged.filter(pl.col("direction") == "Stronger in Pediatric").height
    stronger_adult = inter_merged.filter(pl.col("direction") == "Stronger in Adult").height
    print(f"\nDirection: {stronger_ped:,} stronger in Pediatric | {stronger_adult:,} stronger in Adult")
else:
    print("No intersection data")

**Intersection: 17,157 shared pairs (top 50 by ROR ratio)**

Drug,Event,a (adult),a (ped),ROR adult,ROR ped,ROR ratio,EBGM adult,EBGM ped,Direction
str,str,i64,i64,f64,f64,f64,f64,f64,str
"""EVOLOCUMAB""","""Arteriosclerosis""",8,2,37.73,56059.76,1485.81,36.85,25079.92,"""Stronger in Pediatric"""
"""NATAMYCIN""","""Endophthalmitis""",2,1,14.55,10707.26,735.89,14.5,5294.65,"""Stronger in Pediatric"""
"""VUDALIMAB""","""Hypophysitis""",2,1,49.33,35296.59,715.52,48.98,17018.52,"""Stronger in Pediatric"""
"""AVEED""","""Prostatomegaly""",1,1,49.71,29781.81,599.11,49.51,18687.0,"""Stronger in Pediatric"""
"""HALAVEN""","""Erythropenia""",1,1,37.66,19854.38,527.2,37.49,12707.16,"""Stronger in Pediatric"""
…,…,…,…,…,…,…,…,…,…
"""DESOXIMETASONE""","""Pemphigus""",2498,1,3.3,362.05,109.71,3.2,345.93,"""Stronger in Pediatric"""
"""ADALIMUMAB""","""Neurosarcoidosis""",6,7,4.57,501.14,109.66,4.49,208.84,"""Stronger in Pediatric"""
"""METRONIDAZOLE BENZOATE""","""Appendicitis""",3,1,6.27,687.11,109.59,6.24,457.75,"""Stronger in Pediatric"""



Direction: 10,705 stronger in Pediatric | 6,442 stronger in Adult


## 6. Subgroup Analysis — NICHD Age Bands

คำนวณ signals ใหม่ภายในแต่ละ age band ของ pediatric:
- infancy (0-11 months)
- toddler (1 year)
- early_childhood (2-5 years)
- middle_childhood (6-11 years)
- early_adolescence (12-17 years)
- late_adolescence (18-21 years)

In [212]:
nichd_bands = ["infancy", "toddler", "early_childhood", "middle_childhood",
               "early_adolescence", "late_adolescence"]

# Use top 30 pediatric pairs by ROR for subgroup analysis
top_ped_by_ror = ped_signals.sort('ROR', descending=True, nulls_last=True).head(30)
top_ped_pairs = list(zip(top_ped_by_ror['drug'].to_list(), top_ped_by_ror['event'].to_list()))
subgroup_df = pl.DataFrame()

if "nichd" in pediatric.columns:
    subgroup_dfs = []
    for band in nichd_bands:
        sub_df = pediatric.filter(pl.col("nichd") == band)
        if sub_df.height < 10:
            print(f"  {band}: {sub_df.height} rows — skipped")
            continue

        print(f"  {band}: {sub_df.height:,} rows")
        band_signals = compute_signals_batch(
            sub_df, "medicinal_product", "reaction_meddrapt", top_ped_pairs
        )
        band_signals = band_signals.with_columns(
            pl.lit(band).alias("nichd_band"),
            pl.lit(sub_df.height).alias("subgroup_n"),
        )
        subgroup_dfs.append(band_signals)

    if subgroup_dfs:
        subgroup_df = pl.concat(subgroup_dfs, how="vertical_relaxed")
        print(f"\nTotal: {subgroup_df.height} rows across {len(subgroup_dfs)} bands")
else:
    print("nichd column not available")

  infancy: 64,675 rows
  toddler: 32,239 rows
  early_childhood: 136,785 rows
  middle_childhood: 173,351 rows
  early_adolescence: 317,459 rows
  late_adolescence: 228,528 rows

Total: 180 rows across 6 bands


In [213]:
# Display subgroup results (a >= 3 only) — compact format
if subgroup_df.height > 0:
    sub_rows = []
    for r in subgroup_df.filter(pl.col("a") >= 3).iter_rows(named=True):
        sub_rows.append({
            "NICHD Band": r["nichd_band"],
            "N (subgroup)": r["subgroup_n"],
            "Drug": r["drug"],
            "Event": r["event"],
            "a": r["a"],
            "b": r["b"],
            "c": r["c"],
            "d": r["d"],
            "ROR (95% CI)": _fmt(r["ROR"], r["ROR_lower"], r["ROR_upper"]),
            "PRR (95% CI)": _fmt(r["PRR"], r["PRR_lower"], r["PRR_upper"]),
            "chi2": round(r["chi2"], 2) if r.get("chi2") is not None and not np.isnan(r.get("chi2", float("nan"))) else None,
            "EBGM (95% CI)": _fmt_ebgm(r["EBGM"], r["EBGM05"], r["EBGM95"]),
            "Signal (P/R/E)": _signal_icon(r["PRR_signal"], r["ROR_signal"], r["EBGM_signal"]),
        })
    sub_table = pl.DataFrame(sub_rows).sort(["Drug", "Event", "NICHD Band"])
    display(Markdown(f"**Pediatric Subgroup Signals (a >= 3): {sub_table.height} rows**"))
    display(sub_table)
else:
    print("No subgroup data")

ColumnNotFoundError: unable to find column "Drug"; valid columns: []

## 7. Cross-Cohort Comparison

เปรียบเทียบ drug-event pairs **เดียวกัน** ระหว่าง Adult, Pediatric, และทุก NICHD subgroup

In [ ]:
# Find common drug-event pairs in both cohorts
common_drugs = set(adult["medicinal_product"].unique().to_list()) & set(pediatric["medicinal_product"].unique().to_list())
common_events = set(adult["reaction_meddrapt"].unique().to_list()) & set(pediatric["reaction_meddrapt"].unique().to_list())

common_pairs = [
    (drug, event) for drug, event in adult_pairs
    if drug in common_drugs and event in common_events
][:20]

print(f"Common pairs for comparison: {len(common_pairs)}")

In [ ]:
# Compute for Adult
print("  Adult...")
cross_adult = compute_signals_batch(
    adult, "medicinal_product", "reaction_meddrapt", common_pairs
).with_columns(pl.lit("Adult").alias("cohort"))

# Compute for Pediatric overall
print("  Pediatric...")
cross_ped = compute_signals_batch(
    pediatric, "medicinal_product", "reaction_meddrapt", common_pairs
).with_columns(pl.lit("Pediatric").alias("cohort"))

cross_parts = [cross_adult, cross_ped]

# Compute for each NICHD subgroup
if "nichd" in pediatric.columns:
    for band in nichd_bands:
        sub_df = pediatric.filter(pl.col("nichd") == band)
        if sub_df.height >= 10:
            print(f"  Ped-{band}...")
            cross_sub = compute_signals_batch(
                sub_df, "medicinal_product", "reaction_meddrapt", common_pairs
            ).with_columns(pl.lit(f"Ped-{band}").alias("cohort"))
            cross_parts.append(cross_sub)

cross_df = pl.concat(cross_parts, how="vertical_relaxed")
print(f"\nTotal: {cross_df.height} rows")

In [ ]:
# Display cross-cohort comparison — compact format
display(Markdown("### Cross-Cohort Signal Comparison"))

cross_rows = []
for r in cross_df.iter_rows(named=True):
    cross_rows.append({
        "Cohort": r["cohort"],
        "Drug": r["drug"],
        "Event": r["event"],
        "a": r["a"],
            "b": r["b"],
            "c": r["c"],
            "d": r["d"],
        "Expected (E)": round(r["E (expected)"], 2),
        "ROR (95% CI)": _fmt(r["ROR"], r["ROR_lower"], r["ROR_upper"]),
        "PRR (95% CI)": _fmt(r["PRR"], r["PRR_lower"], r["PRR_upper"]),
        "chi2": round(r["chi2"], 2) if r.get("chi2") is not None and not np.isnan(r.get("chi2", float("nan"))) else None,
        "EBGM (95% CI)": _fmt_ebgm(r["EBGM"], r["EBGM05"], r["EBGM95"]),
        "Signal (P/R/E)": _signal_icon(r["PRR_signal"], r["ROR_signal"], r["EBGM_signal"]),
    })
cross_table = pl.DataFrame(cross_rows).sort(["Drug", "Event", "Cohort"])
display(cross_table)

## 8. Visualization — PRR Comparison

In [ ]:
if cross_df.height > 0:
    cross_with_label = cross_df.with_columns(
        (pl.col("drug") + " + " + pl.col("event")).alias("pair")
    )

    # Filter to signal pairs only
    has_signal = (
        cross_with_label
        .group_by("pair")
        .agg(pl.col("PRR_signal").any().alias("any_signal"))
        .filter(pl.col("any_signal"))["pair"].to_list()
    )

    signal_pairs = cross_with_label.filter(pl.col("pair").is_in(has_signal))

    if signal_pairs.height > 0:
        pivot_data = (
            signal_pairs
            .filter(pl.col("cohort").is_in(["Adult", "Pediatric"]))
            .select(["pair", "cohort", "PRR"])
            .pivot(on="cohort", index="pair", values="PRR")
            .sort("pair")
        )

        if pivot_data.height > 0:
            fig, ax = plt.subplots(figsize=(10, max(4, pivot_data.height * 0.4)))
            pairs_list = pivot_data["pair"].to_list()
            adult_prr = [v if v is not None else 0 for v in pivot_data.get_column("Adult").to_list()]
            ped_prr = [v if v is not None else 0 for v in pivot_data.get_column("Pediatric").to_list()]

            y = np.arange(len(pairs_list))
            h = 0.35
            ax.barh(y - h/2, adult_prr, h, label="Adult PRR", color="#2196F3", alpha=0.8)
            ax.barh(y + h/2, ped_prr, h, label="Pediatric PRR", color="#FF9800", alpha=0.8)
            ax.axvline(x=2, color="red", linestyle="--", alpha=0.5, label="PRR=2 threshold")
            ax.set_yticks(y)
            ax.set_yticklabels(pairs_list, fontsize=8)
            ax.set_xlabel("PRR")
            ax.set_title("PRR Comparison — Adult vs Pediatric (signal pairs only)")
            ax.legend()
            plt.tight_layout()
            plt.savefig("../notebook/prr_comparison.png", dpi=150, bbox_inches="tight")
            plt.show()

## 9. Signal Filtering — Export Only Confirmed Signals

Filter drug-AE pairs where **all 3 methods agree** (PRR=1, ROR=1, EBGM=1):
- This is the most conservative criterion
- Only pairs with robust, confirmed disproportionality signals pass
- Output saved to `data/notebook/output/filtered/` as CSV for downstream analysis

In [ ]:
# ============================================================
# Filter: keep only pairs where ALL signals = 1 (1/1/1)
# ============================================================

FILTER_DIR = Path("../data/notebook/output/disproportionality")
FILTER_DIR.mkdir(parents=True, exist_ok=True)

# Subdirectories
(FILTER_DIR / "adult").mkdir(exist_ok=True)
(FILTER_DIR / "pediatric").mkdir(exist_ok=True)
(FILTER_DIR / "intersection").mkdir(exist_ok=True)

def filter_confirmed_signals(signals_df, label=""):
    """Filter to pairs where PRR, ROR, and EBGM all signal positive."""
    confirmed = signals_df.filter(
        (pl.col("PRR_signal") == True)
        & (pl.col("ROR_signal") == True)
        & (pl.col("EBGM_signal") == True)
        & (pl.col("a") > 0)
        & (pl.col("b") > 0)
        & (pl.col("c") > 0)
        & (pl.col("d") > 0)
    ).sort("ROR", descending=True, nulls_last=True)

    print(f"{label}: {confirmed.height:,} / {signals_df.height:,} pairs passed (1/1/1)")
    return confirmed

# 1. Filter Adult
adult_confirmed = filter_confirmed_signals(adult_signals, "Adult")

# 2. Filter Pediatric
ped_confirmed = filter_confirmed_signals(ped_signals, "Pediatric")

# 3. Filter Intersection — filter EACH side separately, then keep pairs that pass both
inter_adult_confirmed = filter_confirmed_signals(inter_adult_signals, "Intersection (Adult side)")
inter_ped_confirmed = filter_confirmed_signals(inter_ped_signals, "Intersection (Ped side)")

# Intersection confirmed = pairs that pass 1/1/1 in BOTH cohorts
inter_adult_set = set(zip(inter_adult_confirmed["drug"].to_list(), inter_adult_confirmed["event"].to_list()))
inter_ped_set = set(zip(inter_ped_confirmed["drug"].to_list(), inter_ped_confirmed["event"].to_list()))
inter_both_set = inter_adult_set & inter_ped_set

intersection_confirmed = inter_merged.filter(
    pl.struct(["drug", "event"]).map_batches(
        lambda s: pl.Series([
            (r["drug"], r["event"]) in inter_both_set
            for r in s.struct.unnest().iter_rows(named=True)
        ]),
        return_dtype=pl.Boolean,
    )
).sort("ROR_ratio_ped_vs_adult", descending=True, nulls_last=True)

print(f"\nIntersection confirmed (1/1/1 in BOTH): {intersection_confirmed.height:,} pairs")

In [ ]:
# Display Adult confirmed signals
display(Markdown(f"### Adult Confirmed Signals (1/1/1): {adult_confirmed.height:,} pairs"))

adult_display = []
for r in adult_confirmed.head(50).iter_rows(named=True):
    adult_display.append({
        "Drug": r["drug"],
        "RxCUI": r.get("rxcui", ""),
        "Event": r["event"],
        "a": r["a"], "b": r["b"], "c": r["c"], "d": r["d"],
        "Expected (E)": round(r["E (expected)"], 2),
        "ROR (95% CI)": _fmt(r["ROR"], r["ROR_lower"], r["ROR_upper"]),
        "PRR (95% CI)": _fmt(r["PRR"], r["PRR_lower"], r["PRR_upper"]),
        "chi2": round(r["chi2"], 2) if r.get("chi2") is not None and not np.isnan(r.get("chi2", float("nan"))) else None,
        "EBGM (95% CI)": _fmt_ebgm(r["EBGM"], r["EBGM05"], r["EBGM95"]),
    })
display(pl.DataFrame(adult_display))

In [ ]:
# Display Pediatric confirmed signals
display(Markdown(f"### Pediatric Confirmed Signals (1/1/1): {ped_confirmed.height:,} pairs"))

ped_display = []
for r in ped_confirmed.head(50).iter_rows(named=True):
    ped_display.append({
        "Drug": r["drug"],
        "RxCUI": r.get("rxcui", ""),
        "Event": r["event"],
        "a": r["a"], "b": r["b"], "c": r["c"], "d": r["d"],
        "Expected (E)": round(r["E (expected)"], 2),
        "ROR (95% CI)": _fmt(r["ROR"], r["ROR_lower"], r["ROR_upper"]),
        "PRR (95% CI)": _fmt(r["PRR"], r["PRR_lower"], r["PRR_upper"]),
        "chi2": round(r["chi2"], 2) if r.get("chi2") is not None and not np.isnan(r.get("chi2", float("nan"))) else None,
        "EBGM (95% CI)": _fmt_ebgm(r["EBGM"], r["EBGM05"], r["EBGM95"]),
    })
display(pl.DataFrame(ped_display))

### Intersection: Confirmed in BOTH cohorts (1/1/1)

Drug-AE pairs with robust signal in **both** Adult and Pediatric.
Sorted by `ROR_ratio_ped_vs_adult` (>1 = stronger in pediatric).

In [ ]:
# Display intersection — side by side comparison
if intersection_df.height > 0:
    inter_display = []
    for r in intersection_df.head(50).iter_rows(named=True):
        inter_display.append({
            "Drug": r["drug"],
            "RxCUI": r.get("rxcui", ""),
            "Event": r["event"],
            "a (adult)": r["a_adult"],
            "a (ped)": r["a_ped"],
            "ROR adult (95% CI)": _fmt(r["ROR_adult"], r["ROR_lower_adult"], r["ROR_upper_adult"]),
            "ROR ped (95% CI)": _fmt(r["ROR_ped"], r["ROR_lower_ped"], r["ROR_upper_ped"]),
            "ROR ratio (ped/adult)": round(r["ROR_ratio_ped_vs_adult"], 2) if r["ROR_ratio_ped_vs_adult"] is not None else None,
            "EBGM adult (95% CI)": _fmt_ebgm(r["EBGM_adult"], r["EBGM05_adult"], r["EBGM95_adult"]),
            "EBGM ped (95% CI)": _fmt_ebgm(r["EBGM_ped"], r["EBGM05_ped"], r["EBGM95_ped"]),
            "Direction": r["direction"],
        })
    display(Markdown(f"**Intersection Confirmed Signals: {intersection_df.height:,} pairs (top 50 by ROR ratio ped/adult)**"))
    display(pl.DataFrame(inter_display))
    
    # Summary
    stronger_ped = intersection_df.filter(pl.col("direction") == "Stronger in Pediatric").height
    stronger_adult = intersection_df.filter(pl.col("direction") == "Stronger in Adult").height
    print(f"\nDirection: {stronger_ped:,} stronger in Pediatric | {stronger_adult:,} stronger in Adult")
else:
    print("No intersection data")

In [ ]:
# Filter NICHD subgroup — confirmed signals only
if subgroup_df.height > 0:
    subgroup_confirmed = filter_confirmed_signals(subgroup_df, "Subgroup (all bands)")
    
    # Per-band breakdown
    for band in nichd_bands:
        band_data = subgroup_df.filter(pl.col("nichd_band") == band)
        if band_data.height > 0:
            band_confirmed = filter_confirmed_signals(band_data, f"  {band}")
else:
    subgroup_confirmed = pl.DataFrame()
    print("No subgroup data")

In [ ]:
# ============================================================
# Export confirmed signals to CSV — 3 datasets in separate folders
# ============================================================

export_cols = ["drug", "event", "a", "b", "c", "d", "E (expected)",
               "ROR", "ROR_lower", "ROR_upper", "ROR_signal",
               "PRR", "PRR_lower", "PRR_upper", "PRR_signal",
               "chi2",
               "EBGM", "EBGM05", "EBGM95", "EBGM_signal"]

def safe_select(df, cols):
    return df.select([c for c in cols if c in df.columns])

# 1. Adult → data/notebook/output/disproportionality/adult/
adult_csv = safe_select(adult_confirmed, export_cols)
adult_csv.write_csv(FILTER_DIR / "adult" / "confirmed_signals.csv")
adult_signals.write_csv(FILTER_DIR / "adult" / "all_signals.csv")

# 2. Pediatric → data/notebook/output/disproportionality/pediatric/
ped_csv = safe_select(ped_confirmed, export_cols)
ped_csv.write_csv(FILTER_DIR / "pediatric" / "confirmed_signals.csv")
ped_signals.write_csv(FILTER_DIR / "pediatric" / "all_signals.csv")

# 3. Intersection → data/notebook/output/disproportionality/intersection/
if intersection_confirmed.height > 0:
    intersection_confirmed.write_csv(FILTER_DIR / "intersection" / "confirmed_signals.csv")
inter_merged.write_csv(FILTER_DIR / "intersection" / "all_signals.csv")

# 4. Subgroup
if subgroup_confirmed.height > 0:
    (FILTER_DIR / "pediatric" / "subgroup").mkdir(exist_ok=True)
    sub_export = ["nichd_band", "subgroup_n"] + export_cols
    safe_select(subgroup_confirmed, sub_export).write_csv(
        FILTER_DIR / "pediatric" / "subgroup" / "confirmed_signals.csv"
    )

# Summary
print(f"Exported to {FILTER_DIR}/")
print(f"")
print(f"  adult/")
print(f"    confirmed_signals.csv  ({adult_csv.height:,} rows)")
print(f"    all_signals.csv        ({adult_signals.height:,} rows)")
print(f"")
print(f"  pediatric/")
print(f"    confirmed_signals.csv  ({ped_csv.height:,} rows)")
print(f"    all_signals.csv        ({ped_signals.height:,} rows)")
if subgroup_confirmed.height > 0:
    print(f"    subgroup/confirmed_signals.csv ({subgroup_confirmed.height:,} rows)")
print(f"")
print(f"  intersection/")
print(f"    confirmed_signals.csv  ({intersection_confirmed.height:,} rows) — 1/1/1 in BOTH cohorts")
print(f"    all_signals.csv        ({inter_merged.height:,} rows) — all shared pairs")
print(f"")
print(f"--- Summary ---")
print(f"  Adult confirmed:        {adult_confirmed.height:,} / {adult_signals.height:,} ({adult_confirmed.height/adult_signals.height*100:.1f}%)")
print(f"  Pediatric confirmed:    {ped_confirmed.height:,} / {ped_signals.height:,} ({ped_confirmed.height/ped_signals.height*100:.1f}%)")
print(f"  Intersection confirmed: {intersection_confirmed.height:,} / {inter_merged.height:,} ({intersection_confirmed.height/inter_merged.height*100:.1f}%)")

## 10. Export All Results (including unfiltered)

In [ ]:
output_dir = Path("../data/notebook/output/analysis")
output_dir.mkdir(parents=True, exist_ok=True)

adult_signals.write_parquet(output_dir / "adult_disproportionality_signals.parquet")
ped_signals.write_parquet(output_dir / "pediatric_disproportionality_signals.parquet")
cross_df.write_parquet(output_dir / "cross_cohort_comparison.parquet")
if subgroup_df.height > 0:
    subgroup_df.write_parquet(output_dir / "pediatric_nichd_subgroup_signals.parquet")

print(f"Saved to {output_dir}/")
print(f"  adult_disproportionality_signals.parquet     ({adult_signals.height} rows)")
print(f"  pediatric_disproportionality_signals.parquet  ({ped_signals.height} rows)")
print(f"  cross_cohort_comparison.parquet               ({cross_df.height} rows)")
if subgroup_df.height > 0:
    print(f"  pediatric_nichd_subgroup_signals.parquet      ({subgroup_df.height} rows)")

---

## Methodology Notes

| Metric | Formula | SE | Signal Threshold |
|--------|---------|-----|-----------------|
| **ROR** | (a*d) / (b*c) | sqrt(1/a + 1/b + 1/c + 1/d) | 95% CI lower > 1 |
| **PRR** | (a/(a+b)) / (c/(c+d)) | sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d)) | 95% CI lower > 1 |
| **EBGM** | a*N / ((a+c)*(a+b)) | sqrt(1/a + 1/b + 1/c + 1/d) | EBGM05 > 2 |

All 95% CI = exp(ln(metric) +/- 1.96 * SE)

**Subgroup analysis:** 2x2 table reconstructed within each NICHD age band independently.